In [1]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Annotated
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv
from typing import TypedDict

In [3]:
load_dotenv()

True

In [4]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [5]:
class JokeState(TypedDict):
    topic: str
    joke:str
    explanation:str

In [6]:
def generate_joke(state: JokeState):
    prompt = f'Generate a joke on the topic {state['topic']}'
    response = llm.invoke(prompt).content
    
    return {'joke':response}

In [7]:
def generate_explanation(state : JokeState):
    
    prompt = f'Write an explaination for the joke - {state['joke']}'
    
    response = llm.invoke(prompt).content
    
    return {'explanation':response}

In [8]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke',generate_joke)
graph.add_node('generate_explanation',generate_explanation)

graph.add_edge(START,'generate_joke')
graph.add_edge('generate_joke','generate_explanation')
graph.add_edge('generate_explanation',END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [9]:
config1 = {'configurable':{'thread_id':1}}
workflow.invoke({'topic':'Pizza'},config=config1)

{'topic': 'Pizza',
 'joke': 'Why did the pizza get upset?\nBecause it felt like everyone was taking a **pizza** its mind!',
 'explanation': 'This is a classic pun-based joke that plays on a common English idiom!\n\nHere\'s the breakdown:\n\n1.  **The Pun:** The core of the joke is the wordplay between "pizza" and "piece of." They sound very similar when spoken quickly.\n\n2.  **The Idiom:** The joke is referencing the idiom "to **give someone a piece of your mind**" or "to **take a piece of someone\'s mind**."\n    *   **To give someone a piece of your mind** means to express your anger or annoyance very directly and forcefully to them; to scold them.\n    *   **To take a piece of someone\'s mind** (less common, but used here) implies someone else is expressing their anger *to you*.\n\n3.  **Putting It Together:**\n    *   The joke personifies the pizza, giving it human feelings ("upset").\n    *   When the punchline says "everyone was taking a **pizza** its mind," it sounds exactly li

In [10]:
config2 = {'configurable':{'thread_id':2}}
workflow.invoke({'topic':'Pasta'},config=config2)

{'topic': 'Pasta',
 'joke': 'What do you call a fake noodle?\nAn **impasta**!',
 'explanation': 'This joke is a classic **pun**, playing on words that sound alike but have different meanings.\n\nHere\'s the breakdown:\n\n1.  **"Impasta" sounds like "Impostor"**:\n    *   An **impostor** is someone who pretends to be someone else, or something that is fake, not genuine, or misleading.\n    *   The joke asks about a "**fake** noodle," which directly relates to the idea of something being an "impostor."\n\n2.  **"Impasta" contains "Pasta"**:\n    *   **Pasta** is a general term for many types of noodles.\n    *   So, the "noodle" part of the question is covered by "pasta."\n\nBy combining the sound of "impostor" (meaning fake) with "pasta" (meaning noodle), you get "impasta," which perfectly answers "What do you call a fake noodle?" It\'s a clever and silly play on words!'}

In [13]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'Pizza', 'joke': 'Why did the pizza get upset?\nBecause it felt like everyone was taking a **pizza** its mind!', 'explanation': 'This is a classic pun-based joke that plays on a common English idiom!\n\nHere\'s the breakdown:\n\n1.  **The Pun:** The core of the joke is the wordplay between "pizza" and "piece of." They sound very similar when spoken quickly.\n\n2.  **The Idiom:** The joke is referencing the idiom "to **give someone a piece of your mind**" or "to **take a piece of someone\'s mind**."\n    *   **To give someone a piece of your mind** means to express your anger or annoyance very directly and forcefully to them; to scold them.\n    *   **To take a piece of someone\'s mind** (less common, but used here) implies someone else is expressing their anger *to you*.\n\n3.  **Putting It Together:**\n    *   The joke personifies the pizza, giving it human feelings ("upset").\n    *   When the punchline says "everyone was taking a **pizza** its mind," 

# Time Travel

In [15]:
workflow.get_state({'configurable':{'thread_id':1,'checkpoint_id': '1f1d6771-5020-6e28-8000-8ac8cf0f5e2d'}})

StateSnapshot(values={'topic': 'Pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f1d6771-5020-6e28-8000-8ac8cf0f5e2d'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-11-02T04:37:16.710429+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d6771-501a-6ca2-bfff-65594804243c'}}, tasks=(PregelTask(id='4616fae3-20ea-ae5b-39ca-3624072e4d4d', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the pizza get upset?\nBecause it felt like everyone was taking a **pizza** its mind!'}),), interrupts=())

In [16]:
workflow.invoke(None,{'configurable':{'thread_id':1,'checkpoint_id': '1f1d6771-5020-6e28-8000-8ac8cf0f5e2d'}})

{'topic': 'Pizza',
 'joke': 'Why did the pizza cross the road?\nTo get to the other slice!',
 'explanation': 'This is a classic "Why did the chicken cross the road?" style joke, which sets up the expectation for a logical, often absurd, reason for the pizza to travel from one side of the street to the other.\n\nThe humor comes from the **pun** on the word "slice":\n\n1.  **A slice of pizza:** This is the primary association we have with pizza. When you think of pizza, you think of individual slices.\n2.  **"The other side" or "the other part":** The phrase "to get to the other side" is the standard punchline for the "crossing the road" joke. The joke cleverly replaces "the other side" (of the road) with "the other slice," using a term inherently linked to pizza.\n\nSo, the joke plays on the double meaning: the pizza isn\'t trying to get to a *different piece* of pizza, but rather to the "other side" of the road, using a pizza-specific word to describe that "side."\n\nThe humor lies in 

In [17]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'Pizza', 'joke': 'Why did the pizza cross the road?\nTo get to the other slice!', 'explanation': 'This is a classic "Why did the chicken cross the road?" style joke, which sets up the expectation for a logical, often absurd, reason for the pizza to travel from one side of the street to the other.\n\nThe humor comes from the **pun** on the word "slice":\n\n1.  **A slice of pizza:** This is the primary association we have with pizza. When you think of pizza, you think of individual slices.\n2.  **"The other side" or "the other part":** The phrase "to get to the other side" is the standard punchline for the "crossing the road" joke. The joke cleverly replaces "the other side" (of the road) with "the other slice," using a term inherently linked to pizza.\n\nSo, the joke plays on the double meaning: the pizza isn\'t trying to get to a *different piece* of pizza, but rather to the "other side" of the road, using a pizza-specific word to describe that "side."\n

# Updating State

In [20]:
workflow.update_state({'configurable':{'thread_id':1,'checkpoint_id': '1f1d6771-5020-6e28-8000-8ac8cf0f5e2d','checkpoint_ns': ''}},{'topic':'Samasa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1d6794-5c52-64f6-8001-fc790f9389d0'}}

In [21]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'Samasa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d6794-5c52-64f6-8001-fc790f9389d0'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-11-02T04:52:57.513054+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d6771-5020-6e28-8000-8ac8cf0f5e2d'}}, tasks=(PregelTask(id='9b659d10-9057-3645-5949-ecd2cbb964de', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'Pizza', 'joke': 'Why did the pizza cross the road?\nTo get to the other slice!', 'explanation': 'This is a classic "Why did the chicken cross the road?" style joke, which sets up the expectation for a logical, often absurd, reason for the pizza to travel from one side of the street to the other.\n\nThe humor comes from the **pun** on the word "s

In [24]:
workflow.invoke(None,{'configurable':{'thread_id':1,'checkpoint_id': '1f1d6794-5c52-64f6-8001-fc790f9389d0'}})

GoogleRateLimitError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.\nPlease retry in 1h53.166323357s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Gen